# MLSys Task 4：数据、算法与系统优化——如何系统性地优化？

对应打卡 issue：[datawhalechina/llm-algo-leetcode #136](https://github.com/datawhalechina/llm-algo-leetcode/issues/136)

理论材料：

- [Starving the GPU](https://harvard-edge.github.io/cs249r_book_dev/mlsysim/tutorials/04_starving_the_gpu.html)（数据流水线 / Data Wall）
- [Design Space Exploration](https://harvard-edge.github.io/cs249r_book_dev/mlsysim/tutorials/12_design_space_exploration.html)（声明式搜索）
- Module 2: Advanced Single-Node Analysis（选读 PDF：投机解码 / Inference-Time Compute）

三个优化维度 ↔ 三个工具：

| 维度 | 工具 | 关键问题 |
|---|---|---|
| 数据流水线 | `DataModel` / `TransformationModel` | CPU 喂不饱 GPU？ |
| 算法优化 | `ServingModel(draft_model=...)` / `InferenceScalingModel` | 投机解码赚不赚？推理时计算多贵？ |
| 设计空间探索 | `DSE` 引擎 | 千种配置怎么自动找最优？ |

全部实验为解析仿真，Colab CPU runtime 可跑。


In [ ]:
# Colab 每次 new runtime 需要重新安装（约 1 分钟，纯 CPU 即可，不需要 GPU）
%pip install -q "git+https://github.com/harvard-edge/cs249r_book.git@dev#subdirectory=mlsysim"


In [ ]:
import warnings, math
warnings.filterwarnings("ignore")

import mlsysim
from mlsysim import ureg
from mlsysim.core.units import Q_
from mlsysim.solvers import (SingleNodeModel, DataModel, TransformationModel,
                             ServingModel, DistributedModel, EconomicsModel,
                             InferenceScalingModel)
from mlsysim.show import table, info
from mlsysim.engine.dse import DSE
from mlsysim.engine.pipeline import Pipeline

print("mlsysim 版本:", mlsysim.__version__)

resnet50 = mlsysim.Models.Vision.ResNet50
a100, h100 = mlsysim.Hardware.Cloud.A100, mlsysim.Hardware.Cloud.H100
llama8b, llama70b = mlsysim.Models.Language.Llama3_8B, mlsysim.Models.Language.Llama3_70B
llama2_7b = mlsysim.Models.Language.Llama2_7B

gpu_solver   = SingleNodeModel()
data_solver  = DataModel()
xform_solver = TransformationModel()
serve        = ServingModel()


def mag(x, unit=None):
    """统一取数值：Quantity 可选转换单位；裸数字直接返回（兼容不同字段的类型差异）。"""
    if hasattr(x, "magnitude"):
        return x.to(unit).magnitude if unit else x.magnitude
    return x


---
## E1 GPU 纯计算时间：你以为的天花板

训练一步有三个串联阶段：① 存储 I/O → ② CPU 预处理 → ③ GPU 计算。**最慢的一环决定吞吐**。
先单独测出 ③，作为后续对比的天花板。

**预测区**：ResNet-50 在 A100 上 batch=256 的单步训练耗时 ≈ ____ ms？吞吐 ≈ ____ img/s？


In [ ]:
# E1: SingleNodeModel = Engine.solve 的 resolver 封装；is_training=True 含反向传播
profile256 = gpu_solver.solve(resnet50, a100, batch_size=256, precision="fp16",
                              efficiency=0.5, is_training=True)

info("GPU 计算基线（天花板）",
     Model=resnet50.name,
     Hardware=a100.name,
     Step_latency=profile256.latency.to("ms"),
     Throughput=f"{profile256.throughput:.0f} img/s",
     Bottleneck=profile256.bottleneck)


---
## E2 存储 I/O 检查：磁盘 / PCIe 供得上吗？

ImageNet JPEG 平均 ~500 KB。GPU 每步吞掉 `batch × sample` 字节，折算成速率后与硬件数据通路比较。

**预测区**：按 E1 步时间算，数据需求速率 ≈ ____ GB/s；A100 的 PCIe Gen4 x16（32 GB/s）会被打满吗？


In [ ]:
# E2: DataModel —— 比较“需求速率 vs 最慢数据通路”
SAMPLE = Q_("500 KB")
step_s = profile256.latency.to("s").magnitude
demand = Q_(256 * SAMPLE.to("GB").magnitude / step_s, "GB/s")

r = data_solver.solve(workload_data_rate=demand, hardware=a100)
info("存储 I/O 检查（按实际需求）",
     数据需求=f"{demand:.2f}",
     供给=f"{r.supply_bw:.2f}",
     瓶颈链路=r.bottleneck,
     利用率=f"{r.utilization:.1%}",
     是否停顿=r.is_stalled)

# issue 指定场景：给定 6 GB/s，PCIe 扛得住吗？（再顺手试一个超载值）
print()
for rate_str in ["6 GB/s", "40 GB/s"]:
    rr = data_solver.solve(workload_data_rate=Q_(rate_str), hardware=a100)
    print(f"需求 {rate_str:>8}: 供给 {rr.supply_bw:.0f} | 利用率 {rr.utilization:.1%} | "
          f"stalled={rr.is_stalled}")


**E2 说明**：注册表里 `Hardware.Cloud.A100` 的数据通路是 **PCIe Gen4 x16 = 32 GB/s**（NVMe 属于 DGX 整机层，不在加速器对象里）。

所以 6 GB/s 的需求利用率只有约 19%，存储 I/O 不是瓶颈——真正的问题在下一个环节。


---
## E3 CPU 预处理检查：8 个 worker vs 64 个 worker

JPEG 解码 / 裁剪 / 增广都在 CPU 上。典型单 worker 吞吐 ≈ 250 MB/s。

**预测区**：batch=512 时 8 worker（2 GB/s）会让 GPU 挨饿吗？消除瓶颈最少要几个 worker？____


In [ ]:
# E3: TransformationModel —— CPU 变换时间 vs GPU 步时间
prof512 = gpu_solver.solve(resnet50, a100, batch_size=512, precision="fp16",
                           efficiency=0.5, is_training=True)
BS = 512

rows = []
for n in [1, 2, 4, 8, 16, 32, 64]:
    t = xform_solver.solve(batch_size=BS, sample_size_bytes=SAMPLE,
                           cpu_throughput=Q_(f"{n*250} MB/s"),
                           accelerator_step_time=prof512.latency)
    rows.append([n,
                 f"{t.transform_time.to('ms').magnitude:.1f} ms",
                 f"{prof512.latency.to('ms').magnitude:.1f} ms",
                 "CPU 瓶颈" if t.is_bottleneck else "OK",
                 f"{t.accelerator_utilization:.1%}"])

table(["CPU workers", "CPU 变换耗时", "GPU 步时间", "判定", "GPU 利用率"], rows)

# 解析法求最少 worker 数：所需吞吐 = 每步数据量 / GPU 步时间
need_Bps = BS * SAMPLE.to("B").magnitude / prof512.latency.to("s").magnitude
min_workers = math.ceil(need_Bps / 250e6)
print()
print(f"解析解：需要 >= {need_Bps/1e9:.2f} GB/s -> 最少 {min_workers} 个 worker")

# 验证：刚好配足时 CPU 不再是瓶颈
t_min = xform_solver.solve(batch_size=BS, sample_size_bytes=SAMPLE,
                           cpu_throughput=Q_(f"{min_workers*250} MB/s"),
                           accelerator_step_time=prof512.latency)
print(f"验证：{min_workers} workers -> is_bottleneck={t_min.is_bottleneck}, "
      f"GPU 利用率={t_min.accelerator_utilization:.1%}")


**E3 解读 —— 什么是 Data Wall**：

- GPU 利用率 = `GPU步时间 / max(CPU变换时间, GPU步时间)`。CPU 慢一截，GPU 就空转一截——**利用率只有 40% 时问题往往不在 GPU**
- 小批量时 GPU 步时间长，CPU 来得及准备；大批量时 GPU 越来越快、CPU 负载线性涨，瓶颈必然翻转（看上表的 crossover）
- 加 worker 是线性解药，但最终会顶到存储 I/O 或 PCIe（E2 的通路）——所以要**三段一起查**，而不是盯着 nvidia-smi 猜


---
## E4 投机解码：Draft 8B 验证 70B

原理：小 draft 模型一次猜 K=4 个 token（便宜），大 target 一次并行验证（把 memory-bound 的权重读取摊到多个 token 上）。
期望接受 token 数 `E = 1 + α(1−α^K)/(1−α)`，α 为接受率。

**预测区**：α=0.75、K=4 时 ITL 加速比 ≈ ____ x？（先手算 E）


In [ ]:
# E4: ServingModel(draft_model=..., draft_acceptance_rate=...)
SEQ = 2048
base = serve.solve(llama70b, h100, seq_len=SEQ, batch_size=1, precision="fp16")
spec = serve.solve(llama70b, h100, seq_len=SEQ, batch_size=1, precision="fp16",
                   draft_model=llama8b, draft_acceptance_rate=0.75)

K = 4            # mlsysim 内置 SPECULATIVE_GAMMA=4
alpha = 0.75
expected = 1 + alpha * (1 - alpha**K) / (1 - alpha)

itl_base = base.itl.to("ms").magnitude
itl_spec = spec.itl.to("ms").magnitude
print(f"无投机解码 ITL : {itl_base:.2f} ms")
print(f"投机解码   ITL : {itl_spec:.2f} ms")
print(f"加速比          : {itl_base/itl_spec:.2f}x")
print(f"理论期望 token/步 E = {expected:.2f} （粗略上限 ≈ {expected:.1f}x，实际还要扣 draft 开销）")
extra_mem = spec.total_memory_required.to("GB").magnitude - base.total_memory_required.to("GB").magnitude
print(f"代价：显存里要多住一个 draft（+{extra_mem:.1f} GB）")


---
## O1（选做）接受率的权衡：draft 太小 vs 太大

扫 α ∈ [0.5, 0.9]。两个极端都不好：

- **α 小**（draft 与 target 分布差太远，通常因为 draft 太小）：猜 4 个只中零星几个，draft 阶段白跑
- **α → 1 需要 draft 很大**：draft 阶段自身耗时逼近 target，即使全收也没赚头


In [ ]:
# O1: acceptance rate sweep
rows = []
for al in [0.5, 0.6, 0.7, 0.75, 0.8, 0.9]:
    s = serve.solve(llama70b, h100, seq_len=SEQ, batch_size=1, precision="fp16",
                    draft_model=llama8b, draft_acceptance_rate=al)
    itl = s.itl.to("ms").magnitude
    expected = 1 + al * (1 - al**K) / (1 - al)
    rows.append([al, f"{expected:.2f}", f"{itl:.2f} ms", f"{itl_base/itl:.2f}x"])

table(["acceptance rate", "E[token/步]", "有效 ITL", "加速比"], rows)


---
## E5 DSE：三维网格自动搜最优并行配置

不用手写三层 for 循环：声明搜索空间 + 目标函数，交给 `DSE` 引擎穷举（解析仿真毫秒级完成）。
配置：Llama3-70B 训练，256×H100 集群（Research_256，Ethernet 100G），目标最大化吞吐。

**预测区**：最优配置会是“TP 拉满”吗？通信和气泡谁占大头？


In [ ]:
# E5: Design Space Exploration
research = mlsysim.Systems.Clusters.Research_256     # 256 x H100, Ethernet 100G

def eval_config(params):
    pipe = Pipeline([DistributedModel()])
    return pipe.run(model=llama70b, fleet=research,
                    batch_size=params["batch_size"],
                    tp_size=params["tp"], pp_size=params["pp"],
                    precision="fp16", efficiency=0.45, seq_len=2048)

space = {"batch_size": [1, 4, 16, 64], "tp": [1, 2, 4], "pp": [1, 2]}
dse = DSE(space=space, objective="maximize: DistributedModel.effective_throughput")
result = dse.search(eval_config)

bp = result["best_params"]
print(f"最优配置: TP={bp['tp']}, PP={bp['pp']}, Batch={bp['batch_size']} "
      f"-> {result['best_objective']:.1f}")
print()
print(f"{'TP':>3} | {'PP':>2} | {'Batch':>5} | {'吞吐(1/s)':>10} | {'步时(ms)':>9} | {'通信占比':>7} | {'气泡占比':>7}")
print("-" * 66)
for cand in result["top_candidates"][:3]:
    p = cand["params"]
    d = cand["result"]["DistributedModel"]
    step_ms = mag(d.step_latency_total, "ms")
    thr = mag(d.effective_throughput)
    comm_pct = mag(d.communication_latency, "ms") / step_ms
    print(f"{p['tp']:>3} | {p['pp']:>2} | {p['batch_size']:>5} | "
          f"{thr:>10.1f} | {step_ms:>9.1f} | "
          f"{comm_pct:>6.0%} | {d.bubble_fraction:>6.0%}")


**E5 怎么解释最优配置**（对照输出作答）：

- 看 top 配置的三个分量谁在主导：DP AllReduce（跨节点梯度同步）、流水线气泡、还是本地 Roofline 计算；
- 直觉校验：TP 越大 → 每卡分片越小，但 Research_256 的 fabric 是 Ethernet 100G（不是 NVLink），TP AllReduce 的惩罚很重——这就是“TP 尽量限制在 NVLink 域内”的行业经验在数据里的体现；
- 注意 DSE 自动跳过了非法配置（如 TP×PP 不能整除 256 时抛异常即丢弃）——这就是“声明式搜索替代嵌套循环”的价值。


---
## O2（选做）给 DSE 加 SLA 约束

吞吐最大的配置往往延迟爆炸。DSE 支持 `<metric> <op> <threshold>` 语法的硬约束，观察最优解如何移动。


In [ ]:
# O2: SLA-constrained DSE（把结果包装成扁平浮点字段，避免单位歧义）
class Cand:
    def __init__(self, params, d):
        self.params = params
        self.throughput = mag(d.effective_throughput)
        self.step_latency_ms = mag(d.step_latency_total, "ms")

def eval_cand(params):
    d = eval_config(params)["DistributedModel"]
    return Cand(params, d)

for sla_ms in [50, 200, 500, 2000, 10000]:
    dse_sla = DSE(space=space, objective="maximize: throughput",
                  constraints=[f"step_latency_ms < {sla_ms}"])
    try:
        res = dse_sla.search(eval_cand)
        p = res["best_params"]
        print(f"SLA < {sla_ms:>6} ms -> 最优 TP={p['tp']} PP={p['pp']} Batch={p['batch_size']} | "
              f"吞吐 {res['best_objective']:.1f}/s | 步时 {res['best_result'].step_latency_ms:.0f} ms")
    except ValueError:
        print(f"SLA < {sla_ms:>6} ms -> 无可行配置（整个搜索空间都被延迟约束杀死）")


**O2 解读**：

- SLA 太紧 → 全军覆没：“吞吐最大化”和“延迟达标”在给定集群上是**对立目标**，只能靠扩集群或改算法调和
- SLA 逐级放宽时观察最优解迁移：约束紧时被迫选小 batch / 低并行度（步时短但吞吐低）；放宽后立刻跳到大 batch 高吞吐配置——**约束改变的是最优解的位置，而不只是过滤掉几个点**


---
## O3（选做）InferenceScalingModel：o1 式“推理时计算”有多贵

o1-style 模型先生成 K 步隐藏推理（每步 ~50 token）再给答案：`T = TTFT + K × T_step`。
**算法选择直接变成基础设施账单**。


In [ ]:
# O3: K=1/8/32 的隐藏推理成本（K=1 即“无推理计算”的基线）
cot = InferenceScalingModel()
rows, base_t = [], None
for kk in [1, 8, 32]:
    rr = cot.solve(model=llama8b, hardware=h100, reasoning_steps=kk,
                   context_length=2048, precision="fp16")
    total_s = rr.total_reasoning_time.to("s").magnitude
    base_t = base_t or total_s
    rows.append([kk,
                 f"{rr.ttft.to('ms').magnitude:.0f} ms",
                 f"{total_s:.2f} s",
                 rr.tokens_generated,
                 f"{rr.energy_per_query.to('J'):.0f} J",
                 f"{total_s/base_t:.1f}x"])

table(["K 步隐藏推理", "TTFT", "总时延", "生成 tokens", "每查询能耗", "vs K=1"], rows)


**O3 解读**：

- K 步成本 ≈ K × 50 tokens × ITL，TTFT 只是固定的一小块 → 总时延倍数略小于 K 并随 K 趋近 K
- 系统含义：**算力需求从训练转移到推理**。训练是一次性 CAPEX，推理时计算却按每条查询持续烧钱（QPS × 时延 → GPU 数量；详见 Task5 的 9M Question）
- 工程对策预告：路由（简单问题走小模型）、投机解码压 ITL、批处理摊带宽


---
## O4（选做）端到端优化报告骨架：日活 100 万的 LLM 聊天应用

场景：LLaMA-3-8B，平均输入 2K tokens、输出 256 tokens，P99 ITL < 80 ms。
下面四个分析块可直接运行，跑完后把数字填进最后的结论模板。


In [ ]:
# O4-A 场景基线：负载估算 + 单卡服务能力
DAU = 1_000_000
queries_per_user = 10                      # 教学假设
qps_avg = DAU * queries_per_user / 86400
qps_peak = qps_avg * 3                     # 峰均比 3x
print(f"平均 QPS ~= {qps_avg:.0f}，峰值 QPS ~= {qps_peak:.0f}")

ctx = 2048 + 256
base8b = serve.solve(llama8b, h100, seq_len=ctx, batch_size=1, precision="fp16")
itl_ms = base8b.itl.to("ms").magnitude
print(f"单卡 batch=1: TTFT {base8b.ttft.to('ms').magnitude:.0f} ms, "
      f"ITL {itl_ms:.1f} ms -> 输出 256 token 需 {itl_ms*256/1000:.1f} s")


In [ ]:
# O4-B 算法优化：投机解码在该场景的收益
spec8b = serve.solve(llama8b, h100, seq_len=ctx, batch_size=1, precision="fp16",
                     draft_model=llama2_7b, draft_acceptance_rate=0.75)
itl_spec_ms = spec8b.itl.to("ms").magnitude
verdict = "满足" if itl_spec_ms < 80 else "仍不满足"
print(f"投机解码 ITL: {itl_spec_ms:.1f} ms (加速 {itl_ms/itl_spec_ms:.2f}x) "
      f"-> {verdict} P99<80ms 的原始 ITL 预算")


In [ ]:
# O4-C 系统优化：mini-DSE（硬件 x batch），挑满足 ITL 预算的最大吞吐配置
best = None
rows = []
for hw_name, hw in [("A100", a100), ("H100", h100)]:
    for bsz in [1, 8, 32, 64]:
        rr = serve.solve(llama8b, hw, seq_len=ctx, batch_size=bsz, precision="fp16")
        if not rr.feasible:
            continue
        itl_i = rr.itl.to("ms").magnitude
        tok_s = bsz / (itl_i / 1000)             # 每 replica 的 token 吞吐
        rows.append([hw_name, bsz, f"{itl_i:.1f} ms", f"{tok_s:.0f} t/s",
                     "ITL 达标" if itl_i < 80 else "超时"])
        if itl_i < 80 and (best is None or tok_s > best["tok_s"]):
            best = {"hw": hw_name, "bs": bsz, "tok_s": tok_s, "itl": itl_i}

table(["硬件", "batch", "ITL", "token 吞吐/replica", "判定"], rows)

if best:
    replicas = math.ceil(qps_peak * 256 / best["tok_s"])
    print()
    print(f"推荐: {best['hw']} x batch={best['bs']} (ITL {best['itl']:.1f} ms) "
          f"-> 峰值需 ~= {replicas} 张卡")


**O4-D 结论模板**（填空即成文）：

1. **数据流水线**：本场景输入是已 tokenize 的文本，每查询数据量仅 KB 级，Data Wall 不是瓶颈（对照 E2/E3：图像训练才容易被 CPU 卡死）
2. **算法优化**：投机解码将 ITL 从 ____ ms 压到 ____ ms（____x），单查询 256-token 输出从 ____ s 降到 ____ s
3. **系统优化**：满足 P99 ITL<80ms 的最优配置为 ____ × batch=____，单卡 token 吞吐 ____ t/s，峰值需 ____ 卡
4. **综合建议**：硬件选 ____；开启投机解码 + continuous batching；预留 ____% 冗余应对峰谷。（四块运行的数字抄进来即可提交）


---
## 打卡对照清单（issue #136）

**最小打卡（E1–E3）**
- [ ] E1：ResNet-50/A100 纯 GPU 步时间（天花板）
- [ ] E2+E3：存储 I/O 判定、CPU 预处理判定、最少 worker 数、“Data Wall”概念一段话

**学有余力 1（+E4 E5 O1）**
- [ ] E4+O1：基准 ITL、α=0.75 加速比、α 扫描趋势与“draft 太小/太大”的解释
- [ ] E5：Top-3 配置表 + 主导因素分析（算力/内存/通信）

**学有余力 2（+O2 O3 O4）**
- [ ] O2：SLA 加入前后最优配置的变化
- [ ] O3：K=0/8/32 的 TTFT 与总时延 + “算力从训练转向推理”
- [ ] O4：端到端系统优化报告（3–4 页）
